# Data pipeline smoke test

Verifies the end-to-end Lichess → filter → eval/NAG extract → board encoding pipeline on a tiny synthetic PGN. Runs on Kaggle CPU notebooks (no GPU needed).

**To run on Kaggle:**
1. Create a new notebook (https://www.kaggle.com/code → New Notebook).
2. In the right sidebar, open **Settings → Internet** and turn it **On**. Required for the `git clone` in cell 1.
3. `File → Import Notebook → GitHub` and paste the URL of this `.ipynb`, or upload the file directly.
4. Click **Run All**. Total runtime is under a minute on CPU.

See `KAGGLE.md` in the repo root for a fuller walkthrough.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/NipunG2010/stunning-fiesta.git"

if Path("/kaggle/working").exists():
    repo_root = Path("/kaggle/working/stunning-fiesta")
    if not repo_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    subprocess.run(["pip", "install", "-q", "-e", str(repo_root)], check=True)
else:
    repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(repo_root / "src"))
print("repo root:", repo_root)

## 1. Round-trip the encoder on the starting position

In [ ]:
import chess
from chess_rl.encoding import encode_board, encode_move, decode_action, legal_action_mask

board = chess.Board()
planes = encode_board(board)
print("board tensor shape:", planes.shape)
print("legal moves:", legal_action_mask(board).sum())

for move in list(board.legal_moves)[:5]:
    action = encode_move(move, board)
    decoded = decode_action(action, board)
    assert decoded == move
    print(f"  {move.uci()} -> action {action} -> {decoded.uci()}")

## 2. Parse a synthetic Lichess-style PGN

Confirms `iter_games_from_zst`, `filter_game`, and `iter_moves` interoperate. We compress a small inline PGN to .zst in memory so we don't need the 30GB download to validate the pipeline.

In [ ]:
import io
import tempfile
from pathlib import Path

import zstandard as zstd

from chess_rl.data import iter_games_from_zst, filter_game, iter_moves, NAG_BLUNDER

pgn = '''[Event "Rated Blitz game"]
[Site "https://lichess.org/abc"]
[White "alice"]
[Black "bob"]
[Result "1-0"]
[WhiteElo "1600"]
[BlackElo "1620"]
[TimeControl "300+0"]
[Termination "Normal"]

1. e4 { [%eval 0.2] } 1... e5 { [%eval 0.3] } 2. Nf3 { [%eval 0.25] } 2... Nc6 { [%eval 0.3] } 3. Bb5 { [%eval 0.32] } 3... a6 { [%eval 0.35] } 4. Ba4 { [%eval 0.3] } 4... Nf6 { [%eval 0.35] } 5. O-O { [%eval 0.4] } 5... Be7 { [%eval 0.4] } 6. Re1 { [%eval 0.4] } 6... b5 { [%eval 0.45] } 7. Bb3 { [%eval 0.4] } 7... d6 { [%eval 0.5] } 8. c3 { [%eval 0.4] } 8... O-O { [%eval 0.45] } 9. h3 { [%eval 0.4] } 9... Nb8?? { [%eval 5.2] } 10. d4 1-0
'''

tmp = Path(tempfile.gettempdir()) / "smoke.pgn.zst"
with open(tmp, "wb") as f:
    f.write(zstd.ZstdCompressor().compress(pgn.encode("utf-8")))

kept = 0
blunders = 0
for game in iter_games_from_zst(tmp):
    if not filter_game(game, min_elo=1500, max_elo=1700, min_plies=10):
        continue
    kept += 1
    for rec in iter_moves(game):
        if rec.nag == NAG_BLUNDER:
            blunders += 1
            print(f"  blunder: {rec.move.uci()} eval={rec.eval_cp}")

print(f"\nkept {kept} game(s); {blunders} blunder(s)")
assert kept == 1 and blunders == 1

## 3. Encode every position from the kept game

In [ ]:
import numpy as np

(game,) = list(iter_games_from_zst(tmp))
board = game.board()
tensors = []
for rec in iter_moves(game):
    tensors.append(encode_board(board))
    board.push(rec.move)

stacked = np.stack(tensors)
print("per-game tensor stack:", stacked.shape, stacked.dtype)
print("mean piece-plane density:", float(stacked[:, :12].mean()))

## Next steps

When the real Lichess file is available, attach it as a Kaggle Dataset and re-point `iter_games_from_zst` at `/kaggle/input/<dataset-slug>/lichess_db_standard_rated_2024-01.pgn.zst`. Expect a kept rate around 5–10% of all games (Elo band + eval filter), and a blunder rate of roughly 6% among annotated plies (Lichess's own published figure for this rating band).